Example on how to run MUSS algorithm

In [1]:
!pip install faiss-cpu


[notice] A new release of pip is available: 24.1.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys
from pathlib import Path

# Notebook lives at muss/ root, so the muss package's parent is one level up
project_root = str(Path().resolve().parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print("Project root:", project_root)

Project root: /Volumes/Vu/Dropbox/0.Amazon_research/03.projects/muss_diversity_relevance_selection


In [3]:
import numpy as np
import random
from sklearn.metrics.pairwise import pairwise_distances

from muss.src.methods.muss import muss

In [4]:
def sample_data(N, d):
    np.random.seed(42)
    x = np.random.rand(N, d)
    q = np.random.rand(N)
    return x, q

In [5]:
# generate synthetic data
N = 100000
k = 50

print("number of data points N =", N)
print("number of items to be selected k =", k)

x, q = sample_data(N, d=50)

print(x.shape)
print(q.shape)

number of data points N = 100000
number of items to be selected k = 50
(100000, 50)
(100000,)


In [6]:
# run MUSS algorithm
selected_idx, diagnostics = muss(x, q, k, m=100, n_partitions=200)

print(diagnostics)

{'time_within': 2.099483013153076, 'time_union': 0.04512214660644531, 'size_union': 5025, 'clustering_time': 1.65}


In [7]:
# this is the selected indices
print(selected_idx)

# average quality score of the selected points
print("average quality = ", np.mean( q[selected_idx]) )

# average pairwise distance of the selected point
pw_x = pairwise_distances(x[selected_idx,:], x[selected_idx,:], metric="euclidean")

print("average pairwise distance = ", np.mean(pw_x))

[55259, 8346, 15378, 11750, 93714, 30252, 93180, 25233, 78652, 12584, 49870, 48026, 25652, 59072, 75329, 60920, 69454, 51118, 38136, 14811, 24349, 68218, 802, 91398, 5994, 32280, 14343, 82200, 62458, 64724, 57554, 64889, 84307, 38564, 87561, 83631, 73544, 41786, 21036, 47886, 25569, 13110, 43171, 36901, 78688, 8002, 39445, 28498, 12908, 79730]
average quality =  0.9692155104223992
average pairwise distance =  3.269116264415774


Random baseline selection

In [8]:
selected_random_indices = random.sample(range(N), k)

In [9]:
import pandas as pd

muss_quality = np.mean(q[selected_idx])
muss_pw_dist = np.mean(pairwise_distances(x[selected_idx, :], x[selected_idx, :], metric="euclidean"))

random_quality = np.mean(q[selected_random_indices])
random_pw_dist = np.mean(pairwise_distances(x[selected_random_indices, :], x[selected_random_indices, :], metric="euclidean"))

pd.DataFrame(
    {
        "Method":            ["MUSS", "Random"],
        "Avg Quality":       [round(muss_quality, 4), round(random_quality, 4)],
        "Avg Pairwise Dist": [round(muss_pw_dist, 4), round(random_pw_dist, 4)],
    }
).set_index("Method")

,Avg Quality,Avg Pairwise Dist
Method,,
MUSS,0.9692,3.2691
Random,0.4654,2.8019
